# Phase 4: Background Stress Test

This notebook calls the Phase 4 stress-test script.

AwA2 has no real segmentation masks, so this phase uses explicit approximate masks:

- `center_ellipse`: preserve the central object-like region and perturb the outside;
- `center_box`: preserve a central box and perturb the outside;
- `global`: perturb the whole image as a fallback.

The goal is to check whether predictions are stable when context/background is changed.

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

CANDIDATE_MANIFESTS = [
    PROJECT_ROOT / "data" / "AWA2_subset_background20" / "awa2_manifest_subset.csv",
    PROJECT_ROOT / "data" / "AWA2" / "awa2_manifest_debug.csv",
    PROJECT_ROOT / "data" / "AWA2" / "awa2_manifest.csv",
]
MANIFEST = next((path for path in CANDIDATE_MANIFESTS if path.exists()), None)
if MANIFEST is None:
    raise FileNotFoundError("No AwA2 manifest found.")

CHECKPOINT = PROJECT_ROOT / "outputs" / "checkpoints" / "best_resnet50_awa2.pt"
FIGURE_OUTPUT = PROJECT_ROOT / "outputs" / "figures" / "phase4_stress_test_notebook.png"
CSV_OUTPUT = PROJECT_ROOT / "outputs" / "reports" / "phase4_stress_test_notebook.csv"

MASK_STRATEGY = "center_ellipse"  # center_ellipse, center_box, global
FOREGROUND_SCALE = 0.68
MAX_IMAGES = 6
ALLOW_INCORRECT = False

print("manifest:", MANIFEST)
print("checkpoint:", CHECKPOINT)
print("checkpoint_exists:", CHECKPOINT.exists())
print("figure_output:", FIGURE_OUTPUT)
print("csv_output:", CSV_OUTPUT)

## Run Stress Test

This command applies three perturbations:

- Gaussian noise on approximate background pixels;
- RGB inversion on approximate background pixels;
- background replacement with uniform random noise.

In [ ]:
cmd = [
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "run_stress_test.py"),
    "--manifest", str(MANIFEST),
    "--checkpoint", str(CHECKPOINT),
    "--figure-output", str(FIGURE_OUTPUT),
    "--csv-output", str(CSV_OUTPUT),
    "--max-images", str(MAX_IMAGES),
    "--mask-strategy", MASK_STRATEGY,
    "--foreground-scale", str(FOREGROUND_SCALE),
]
if ALLOW_INCORRECT:
    cmd.append("--allow-incorrect")

print(" ".join(cmd))
completed = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)
print(completed.stdout)
print(completed.stderr)
if completed.returncode != 0:
    raise RuntimeError(f"Phase 4 failed with exit code {completed.returncode}")

## Inspect Outputs

The figure shows original images, approximate background masks, and the three perturbation variants. The CSV stores prediction changes and confidence deltas.

In [ ]:
print("figure:", FIGURE_OUTPUT)
print("csv:", CSV_OUTPUT)
if FIGURE_OUTPUT.exists():
    display(Image(filename=str(FIGURE_OUTPUT)))
else:
    print("Figure was not created.")

if CSV_OUTPUT.exists():
    print(CSV_OUTPUT.read_text(encoding="utf-8").splitlines()[:10])
else:
    print("CSV was not created.")

## Fallback: Global Perturbation

If the approximate foreground mask is not meaningful for some images, change:

```python
MASK_STRATEGY = "global"
```

and rerun the notebook. This no longer claims background isolation, but still tests whether saliency/prediction behavior is fragile under input perturbations.